# 🌧️ Project 10 — Compare Water Before and After Rain

## 🎯 Goal

In this project, you will compare water conditions on **two different dates**:

- 🌧️ Before rainfall
- 🌊 After rainfall

Then, you'll calculate **NDWI (Normalized Difference Water Index)** for each date and compare them to see how water bodies changed.

By the end of this project, you'll be able to:

- Load satellite images from two different dates.
- Calculate NDWI for each image.
- Compare water conditions over time.
- Create a **water change map** using pixel-by-pixel subtraction.

---

# Why is this useful?

Water bodies change over time because of:

- 🌧️ Rainfall
- 🌊 Floods
- ☀️ Drought
- 💧 Reservoir management
- ❄️ Snowmelt

Comparing images from different dates is called **change detection**, one of the most common tasks in remote sensing.

---

# New Concept: Change Detection

So far, your workflow looked like this:

```
One Image
     ↓
Calculate NDWI
     ↓
Water Map
```

Now we'll compare two images:

```
Before Rain
      ↓
Calculate NDWI
      ↓

After Rain
      ↓
Calculate NDWI
      ↓

Compare
```

Instead of asking:

> "Where is the water?"

we're asking:

> **"How has the water changed?"**

---

# Step 1 — Create a Study Area

We'll use the same polygon as in previous projects.

```javascript
var cityArea = ee.Geometry.Polygon([
  [
    [51.55,32.55],
    [51.80,32.55],
    [51.80,32.75],
    [51.55,32.75],
    [51.55,32.55]
  ]
]);
```

Nothing new here.

This polygon defines the area where we'll search for satellite images.

---

# Step 2 — Load the Sentinel-2 Collection

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);
```

Notice something different.

We are **not selecting an image immediately.**

Why?

Because we need **two different images**:

- one before rainfall
- one after rainfall

The collection acts like a library that stores many Sentinel-2 images taken on different dates.

---

# Step 3 — Create an NDWI Function ⭐

Instead of writing the NDWI calculation twice, we'll create a reusable function.

```javascript
function calculateNDWI(image){
  return image
    .normalizedDifference(['B3','B8'])
    .rename('NDWI');
}
```

Let's break it down.

### `function`

Creates a reusable block of code.

### `calculateNDWI`

The function name.

You can call it whenever you want to calculate NDWI.

### `(image)`

The input.

The function expects an `ee.Image`.

### `return`

Returns the result back to the program.

### `.normalizedDifference(['B3','B8'])`

Calculates:

\[
NDWI=\frac{Green-NIR}{Green+NIR}
\]

For Sentinel-2:

- **B3** → Green
- **B8** → Near Infrared (NIR)

### `.rename('NDWI')`

Changes the output band name from the default (`nd`) to a clearer name (`NDWI`).

---

# Step 4 — Load the Image Before Rain

```javascript
var beforeRain = collection
  .filterBounds(cityArea)
  .filterDate('2025-03-01','2025-03-31')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(cityArea);
```

Review the workflow:

- **filterBounds()** → Keep only images covering the study area.
- **filterDate()** → Select images from March.
- **sort()** → Order images by cloud cover.
- **first()** → Choose the least cloudy image.
- **clip()** → Keep only the study area.

---

# Step 5 — Load the Image After Rain

```javascript
var afterRain = collection
  .filterBounds(cityArea)
  .filterDate('2025-04-01','2025-04-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(cityArea);
```

This follows exactly the same workflow.

Only the **date range changes**.

> **Note:** For a real flood or rainfall study, you would choose dates based on an actual rainfall event. For this learning project, we're simply using two different periods to practice the workflow.

---

# Step 6 — Calculate NDWI

Now use the function you created.

```javascript
var beforeNDWI = calculateNDWI(beforeRain);

var afterNDWI = calculateNDWI(afterRain);
```

Instead of writing:

```javascript
image.normalizedDifference(...)
```

twice, we reuse our function.

This keeps the code cleaner and easier to maintain.

---

# Step 7 — Create an NDWI Visualization

```javascript
var ndwiVis = {
  min:-1,
  max:1,
  palette:[
    'brown',
    'white',
    'blue'
  ]
};
```

### NDWI values

| NDWI | Interpretation |
|------|----------------|
| Negative | Land |
| Around 0 | Mixed surfaces |
| Positive | Water |

### Color palette

- 🟤 Brown → Land
- ⚪ White → Mixed surfaces
- 🔵 Blue → Water

---

# Step 8 — Display Both NDWI Maps

```javascript
Map.centerObject(cityArea,10);

Map.addLayer(
  beforeNDWI,
  ndwiVis,
  'Before Rain'
);

Map.addLayer(
  afterNDWI,
  ndwiVis,
  'After Rain'
);
```

Turn the layers on and off to compare them.

Ask yourself:

- Did the lake become larger?
- Did rivers widen?
- Did new water appear?
- Did some water disappear?

---

# ⭐ Bonus Challenge — Create a Water Change Map

Instead of only comparing the layers visually, we can calculate the difference.

```javascript
var waterChange = afterNDWI.subtract(beforeNDWI);
```

---

## What does `subtract()` do?

`subtract()` performs **pixel-by-pixel subtraction**.

Suppose one pixel has these values:

| Date | NDWI |
|------|------|
| Before | 0.20 |
| After | 0.55 |

Earth Engine calculates:

```
0.55 - 0.20 = +0.35
```

A positive value means:

➡️ Water increased.

Another example:

| Date | NDWI |
|------|------|
| Before | 0.40 |
| After | 0.10 |

```
0.10 - 0.40 = -0.30
```

A negative value means:

➡️ Water decreased.

The result is another **ee.Image**, where every pixel represents the change between the two dates.

---

# Visualize the Change

```javascript
var changeVis = {
  min:-0.5,
  max:0.5,
  palette:[
    'red',
    'white',
    'blue'
  ]
};

Map.addLayer(
  waterChange,
  changeVis,
  'Water Change'
);
```

### Interpretation

- 🔴 Red → Water decreased
- ⚪ White → Little or no change
- 🔵 Blue → Water increased

---

# Complete Code

```javascript
// 1. Study Area
var cityArea = ee.Geometry.Polygon([
  [
    [51.55,32.55],
    [51.80,32.55],
    [51.80,32.75],
    [51.55,32.75],
    [51.55,32.55]
  ]
]);

// 2. Image Collection
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);

// 3. NDWI Function
function calculateNDWI(image){
  return image
    .normalizedDifference(['B3','B8'])
    .rename('NDWI');
}

// 4. Before Rain
var beforeRain = collection
  .filterBounds(cityArea)
  .filterDate('2025-03-01','2025-03-31')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(cityArea);

// 5. After Rain
var afterRain = collection
  .filterBounds(cityArea)
  .filterDate('2025-04-01','2025-04-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(cityArea);

// 6. Calculate NDWI
var beforeNDWI = calculateNDWI(beforeRain);
var afterNDWI = calculateNDWI(afterRain);

// 7. Visualization
var ndwiVis = {
  min:-1,
  max:1,
  palette:['brown','white','blue']
};

// 8. Display
Map.centerObject(cityArea,10);

Map.addLayer(beforeNDWI, ndwiVis, 'Before Rain');
Map.addLayer(afterNDWI, ndwiVis, 'After Rain');

// 9. Bonus: Water Change
var waterChange = afterNDWI.subtract(beforeNDWI);

var changeVis = {
  min:-0.5,
  max:0.5,
  palette:['red','white','blue']
};

Map.addLayer(waterChange, changeVis, 'Water Change');
```

---

# Things to Remember ⭐

## 1. This is your first change detection project

General workflow:

```
Image A
    ↓
Calculate Index
    ↓

Image B
    ↓
Calculate Index
    ↓

Compare
```

This same idea is used for:

- 🌿 NDVI change (vegetation)
- 🌊 NDWI change (water)
- 🔥 Burn severity
- 🏙️ Urban expansion
- ❄️ Glacier retreat

---

## 2. `subtract()` Performs Pixel-by-Pixel Subtraction

```javascript
afterNDWI.subtract(beforeNDWI)
```

means:

```
Every Pixel

After − Before
```

The output is another **ee.Image**, where each pixel stores the amount of change.

---

## 3. Functions Reduce Repetition

You've now created two reusable functions:

```javascript
calculateNDVI(image)

calculateNDWI(image)
```

Whenever you find yourself writing the same code repeatedly, consider creating a function.

This follows the same programming principle you learned in Python.

---

# Common Beginner Mistakes

### ❌ Mistake 1 — Reversing the subtraction order

```javascript
beforeNDWI.subtract(afterNDWI)
```

This flips the sign of the results.

If you want to measure **change after rainfall**, use:

```javascript
afterNDWI.subtract(beforeNDWI)
```

---

### ❌ Mistake 2 — Comparing Different Locations

Always use the same study area:

```javascript
.filterBounds(cityArea)
```

Otherwise, you're comparing different places instead of changes through time.

---

### ❌ Mistake 3 — Ignoring Cloud Cover

Clouds can affect NDWI values.

Choose images with low cloud cover—or apply cloud masking—before comparing dates.

---

# 📝 Your Practice (Project 10.1)

Create your own **water change analysis**.

### Requirements

- ✅ Create a study area.
- ✅ Load the Sentinel-2 collection.
- ✅ Write a `calculateNDWI()` function.
- ✅ Select a **before** image.
- ✅ Select an **after** image.
- ✅ Calculate NDWI for both images.
- ✅ Display both NDWI maps.
- ⭐ **Bonus:** Create a `waterChange` image using `.subtract()` and display it.

---

## Before Writing the Code

### Question 1

Why do we calculate **NDWI first** and then subtract the two NDWI images, instead of subtracting the original Sentinel-2 images?

---

### Question 2

If a pixel changes from:

- Before: **0.15**
- After: **0.55**

What will:

```javascript
afterNDWI.subtract(beforeNDWI)
```

return?

What does that tell you about that location?